# Preparación y limpieza de los datos de **tratamiento** de residuos


## 1. Carga de datos

In [1]:
import pandas as pd
import numpy as np

In [2]:
tratamiento = pd.read_csv('../Data/Original/env_wastrt_linear.csv.gz', compression='gzip')

In [3]:
tratamiento.head()

,DATAFLOW,LAST UPDATE,freq,unit,hazard,wst_oper,waste,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:ENV_WASTRT(1.0),28/10/26 11:00:00,Annual,Kilograms per capita,Hazardous,Disposal - incineration (D10),Other disposed wastes (valid up to 2008),Iceland,2014,NaN,b,NaN
1,ESTAT:ENV_WASTRT(1.0),28/10/26 11:00:00,Annual,Kilograms per capita,Hazardous,Disposal - incineration (D10),Other incinerated wastes (valid up to 2008),Austria,2004,2.0,NaN,NaN
2,ESTAT:ENV_WASTRT(1.0),28/10/26 11:00:00,Annual,Kilograms per capita,Hazardous,Disposal - incineration (D10),Other incinerated wastes (valid up to 2008),Austria,2006,2.0,NaN,NaN
3,ESTAT:ENV_WASTRT(1.0),28/10/26 11:00:00,Annual,Kilograms per capita,Hazardous,Disposal - incineration (D10),Other incinerated wastes (valid up to 2008),Austria,2008,NaN,NaN,C
4,ESTAT:ENV_WASTRT(1.0),28/10/26 11:00:00,Annual,Kilograms per capita,Hazardous,Disposal - incineration (D10),Other incinerated wastes (valid up to 2008),Belgium,2006,0.0,NaN,NaN


In [4]:
tratamiento.info()

<class 'pandas.DataFrame'>
RangeIndex: 595857 entries, 0 to 595856
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATAFLOW     595857 non-null  str    
 1   LAST UPDATE  595857 non-null  str    
 2   freq         595857 non-null  str    
 3   unit         595857 non-null  str    
 4   hazard       595857 non-null  str    
 5   wst_oper     595857 non-null  str    
 6   waste        595857 non-null  str    
 7   geo          595857 non-null  str    
 8   TIME_PERIOD  595857 non-null  int64  
 9   OBS_VALUE    497816 non-null  float64
 10  OBS_FLAG     96376 non-null   str    
 11  CONF_STATUS  11581 non-null   str    
dtypes: float64(1), int64(1), str(10)
memory usage: 54.6 MB


## 2. Primera "limpieza"
Quitamos las columnas que no aportan nada y renombramos algunas de las columnas.

In [5]:
tratamiento_clean = tratamiento.copy()

# Eliminar columnas innecesarias
tratamiento_clean = tratamiento_clean.drop(
    columns=['DATAFLOW', 'LAST UPDATE', 'freq']
)

# Renombrar columnas
tratamiento_clean = tratamiento_clean.rename(columns={
    'TIME_PERIOD': 'year',
    'OBS_VALUE': 'value',
    'OBS_FLAG': 'obs_flag',
    'CONF_STATUS': 'conf_status',
    "wst_oper": "operacion"
})

Transformamos la columna "unit" en dos columnas: "kg_per_capita" y "toneladas".

In [6]:
# Columnas que identifican cada observación
index_cols = [
    'hazard',
    'operacion',
    'waste',
    'geo',
    'year',
    'obs_flag',
    'conf_status'
]

tratamiento_clean = tratamiento_clean.pivot(
    index=index_cols,
    columns='unit',
    values='value'
).reset_index()

# Eliminar el nombre del índice de columnas
tratamiento_clean.columns.name = None

In [7]:
# Renombramos las nuevas columnas
tratamiento_clean = tratamiento_clean.rename(columns={
    'Kilograms per capita': 'kg_per_capita',
    'Tonne': 'toneladas',
})

In [8]:
tratamiento_clean.head()

,hazard,operacion,waste,geo,year,obs_flag,conf_status,kg_per_capita,toneladas
0,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Albania,2024,NaN,NaN,0.0,0.0
1,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2010,NaN,NaN,0.0,44.0
2,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2012,NaN,C,NaN,NaN
3,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2014,NaN,C,NaN,NaN
4,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2016,NaN,C,NaN,NaN


# 3. Filtrado de datos
Filtramos los datos a analizar, manteniendo sólo los países de la UE.

In [9]:
paises_ue = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", 
    "Czechia", "Denmark", "Estonia", "Finland", 
    "France", "Germany", "Greece", "Hungary", "Ireland", 
    "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", 
    "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", 
    "Slovenia", "Spain", "Sweden", 'European Union - 27 countries (from 2020)'
]

tratamiento_clean = tratamiento_clean[tratamiento_clean['geo'].isin(paises_ue)]

In [10]:
tratamiento_clean.head()

,hazard,operacion,waste,geo,year,obs_flag,conf_status,kg_per_capita,toneladas
1,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2010,NaN,NaN,0.0,44.0
2,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2012,NaN,C,NaN,NaN
3,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2014,NaN,C,NaN,NaN
4,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2016,NaN,C,NaN,NaN
5,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2018,NaN,C,NaN,NaN


Además, quitamos los tipos de residuos que se han quedado obsoletos *(up to 2008)*.

In [11]:
tratamiento_clean['waste'].unique()

<StringArray>
[                                                                              'Acid, alkaline or saline wastes',
                                                                                   'Animal and mixed food waste',
                                                                               'Animal faeces, urine and manure',
                                                                             'Batteries and accumulators wastes',
                                                                        'Chemical and medical wastes (subtotal)',
                                                                                               'Chemical wastes',
                                                                                             'Combustion wastes',
                                                                                                'Common sludges',
                                               'Common sludges and dredgin

In [12]:
tratamiento_clean = tratamiento_clean[
    ~tratamiento_clean['waste'].str.contains('valid up to 2008', case=False, na=False)
]

## 4. Traducción de etiquetas
Se conservan las columnas originales (`geo`, `waste`, etc.) y se crean columnas en castellano.

In [13]:
PAISES_ES = {'Austria': 'Austria', 'Belgium': 'Bélgica', 'Bulgaria': 'Bulgaria', 'Croatia': 'Croacia', 'Cyprus': 'Chipre', 'Czechia': 'Chequia', 'Denmark': 'Dinamarca', 'Estonia': 'Estonia', 'Finland': 'Finlandia', 'France': 'Francia', 'Germany': 'Alemania', 'Greece': 'Grecia', 'Hungary': 'Hungría', 'Ireland': 'Irlanda', 'Italy': 'Italia', 'Latvia': 'Letonia', 'Lithuania': 'Lituania', 'Luxembourg': 'Luxemburgo', 'Malta': 'Malta', 'Netherlands': 'Países Bajos', 'Poland': 'Polonia', 'Portugal': 'Portugal', 'Romania': 'Rumanía', 'Slovakia': 'Eslovaquia', 'Slovenia': 'Eslovenia', 'Spain': 'España', 'Sweden': 'Suecia', 'European Union - 27 countries (from 2020)': 'Unión Europea (UE-27)'}

In [14]:
PELIGROSIDAD_ES = {'Hazardous': 'Peligrosos', 'Non-hazardous': 'No peligrosos', 'Hazardous and non-hazardous - total': 'Total (peligrosos y no peligrosos)'}

In [15]:
RESIDUOS_ES = {
    'Acid, alkaline or saline wastes': 'Residuos ácidos, alcalinos o salinos',
    'Animal and mixed food waste': 'Residuos animales y mezclados de alimentos',
    'Animal faeces, urine and manure': 'Heces de animales, orina y estiércol',
    'Batteries and accumulators wastes': 'Residuos de pilas y acumuladores',
    'Chemical and medical wastes (subtotal)': 'Residuos químicos y médicos (subtotal)',
    'Chemical wastes': 'Residuos químicos',
    'Combustion wastes': 'Residuos de combustión',
    'Common sludges': 'Lodos comunes',
    'Discarded equipment (except discarded vehicles and batteries and accumulators waste) (W08 except W081, W0841)': 'Equipos desechados (excepto vehículos desechados y residuos de pilas y acumuladores) (W08 excepto W081, W0841)',
    'Discarded vehicles': 'Vehículos desechados',
    'Dredging spoils': 'Lodos de dragado',
    'Equipment (subtotal, W077+W08A+W081+W0841)': 'Equipos (subtotal, W077+W08A+W081+W0841)',
    'Glass wastes': 'Residuos de vidrio',
    'Health care and biological wastes': 'Residuos sanitarios y biológicos',
    'Household and similar wastes': 'Residuos domésticos y similares',
    'Industrial effluent sludges': 'Lodos de efluentes industriales',
    'Metal wastes, ferrous': 'Residuos metálicos férreos',
    'Metal wastes, mixed ferrous and non-ferrous': 'Residuos metálicos mezclados férreos y no férreos',
    'Metal wastes, non-ferrous': 'Residuos metálicos no férreos',
    'Metallic wastes (W061+W062+W063)': 'Residuos metálicos (W061+W062+W063)',
    'Mineral and solidified wastes (subtotal)': 'Residuos minerales y solidificados (subtotal)',
    'Mineral waste from construction and demolition': 'Residuos minerales de construcción y demolición',
    'Mineral wastes from waste treatment and stabilised wastes': 'Residuos minerales del tratamiento de residuos y residuos estabilizados',
    'Mixed and undifferentiated materials': 'Materiales mezclados e indiferenciados',
    'Mixed ordinary wastes (subtotal, W101+W102+W103)': 'Residuos ordinarios mezclados (subtotal, W101+W102+W103)',
    'Other mineral wastes (W122+W123+W125)': 'Otros residuos minerales (W122+W123+W125)',
    'Paper and cardboard wastes': 'Residuos de papel y cartón',
    'Plastic wastes': 'Residuos de plástico',
    'Primary waste (TOTAL minus SEC)': 'Residuos primarios (TOTAL menos SEC)',
    'Recyclable wastes (subtotal, W06+W07 except W077)': 'Residuos reciclables (subtotal, W06+W07 excepto W077)',
    'Rubber wastes': 'Residuos de caucho',
    'Secondary waste (W033+W103+W128_13)': 'Residuos secundarios (W033+W103+W128_13)',
    'Sludges and liquid wastes from waste treatment': 'Lodos y residuos líquidos del tratamiento de residuos',
    'Soils': 'Tierras y suelos',
    'Sorting residues': 'Residuos de clasificación',
    'Spent solvents': 'Disolventes usados',
    'Textile wastes': 'Residuos textiles',
    'Total waste': 'Total de residuos',
    'Used oils': 'Aceites usados',
    'Vegetal wastes': 'Residuos vegetales',
    'Waste containing PCB': 'Residuos que contienen PCB',
    'Waste excluding major mineral wastes': 'Residuos excluidos los principales residuos minerales',
    'Wood wastes': 'Residuos de madera',
    'Animal and mixed food waste; vegetal wastes (W091+W092)': 'Residuos animales y mezclados de alimentos; residuos vegetales (W091+W092)',
    'Animal and vegetal wastes (subtotal, W091+W092+W093)': 'Residuos animales y vegetales (subtotal, W091+W092+W093)'
}

In [16]:
tratamiento_clean["operacion"].unique()

<StringArray>
[                'Disposal - incineration (D10)',
             'Disposal - landfill (D1, D5, D12)',
    'Disposal - landfill and other (D1-D7, D12)',
               'Disposal - other (D2-D4, D6-D7)',
                        'Recovery - backfilling',
               'Recovery - energy recovery (R1)',
                          'Recovery - recycling',
 'Recovery - recycling and backfilling (R2-R11)',
                               'Waste treatment']
Length: 9, dtype: str

In [17]:
TRATAMIENTOS__ES = {
    'Disposal - incineration (D10)': 'Eliminación - incineración (D10)',
    'Disposal - landfill (D1, D5, D12)': 'Eliminación - depósito en vertedero (D1, D5, D12)',
    'Disposal - landfill and other (D1-D7, D12)': 'Eliminación - depósito en vertedero y otros (D1-D7, D12)',
    'Disposal - other (D2-D4, D6-D7)': 'Eliminación - otros (D2-D4, D6-D7)',
    'Recovery - backfilling': 'Valorización - relleno',
    'Recovery - energy recovery (R1)': 'Valorización - valorización energética (R1)',
    'Recovery - recycling': 'Valorización - reciclado',
    'Recovery - recycling and backfilling (R2-R11)': 'Valorización - reciclado y relleno (R2-R11)',
    'Waste treatment': 'Tratamiento de residuos'
}

In [18]:
tratamiento_clean = tratamiento_clean.copy()

tratamiento_clean['pais'] = tratamiento_clean['geo'].replace(PAISES_ES)
tratamiento_clean['peligrosidad'] = tratamiento_clean['hazard'].replace(PELIGROSIDAD_ES)
tratamiento_clean['residuo'] = tratamiento_clean['waste'].replace(RESIDUOS_ES)
tratamiento_clean['tratamiento'] = tratamiento_clean['operacion'].replace(TRATAMIENTOS__ES)

In [19]:
tratamiento_clean.head()

,hazard,operacion,waste,geo,year,obs_flag,conf_status,kg_per_capita,toneladas,pais,peligrosidad,residuo,tratamiento
1,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2010,NaN,NaN,0.0,44.0,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos",Eliminación - incineración (D10)
2,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2012,NaN,C,NaN,NaN,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos",Eliminación - incineración (D10)
3,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2014,NaN,C,NaN,NaN,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos",Eliminación - incineración (D10)
4,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2016,NaN,C,NaN,NaN,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos",Eliminación - incineración (D10)
5,Hazardous,Disposal - incineration (D10),"Acid, alkaline or saline wastes",Austria,2018,NaN,C,NaN,NaN,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos",Eliminación - incineración (D10)


# 5. Limpieza final y exportación
Para exportar los datos, nos quedamos con los datos en castellano.

In [20]:
columnas_a_mantener = [
    'pais', 
    'year', 
    'peligrosidad', 
    'tratamiento', 
    'residuo', 
    'kg_per_capita', 
    'toneladas', 
    'obs_flag', 
    'conf_status'
]

trt_residuos = tratamiento_clean[columnas_a_mantener]

In [21]:
trt_residuos.head()

,pais,year,peligrosidad,tratamiento,residuo,kg_per_capita,toneladas,obs_flag,conf_status
1,Austria,2010,Peligrosos,Eliminación - incineración (D10),"Residuos ácidos, alcalinos o salinos",0.0,44.0,NaN,NaN
2,Austria,2012,Peligrosos,Eliminación - incineración (D10),"Residuos ácidos, alcalinos o salinos",NaN,NaN,NaN,C
3,Austria,2014,Peligrosos,Eliminación - incineración (D10),"Residuos ácidos, alcalinos o salinos",NaN,NaN,NaN,C
4,Austria,2016,Peligrosos,Eliminación - incineración (D10),"Residuos ácidos, alcalinos o salinos",NaN,NaN,NaN,C
5,Austria,2018,Peligrosos,Eliminación - incineración (D10),"Residuos ácidos, alcalinos o salinos",NaN,NaN,NaN,C


In [22]:
trt_residuos.to_csv("../Data/trt_residuos.csv", index=False)